In [6]:
import pandas as pd
import os
import yaml
from pathlib import Path
import sys
import importlib

%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))             

with open(PROJECT_ROOT / "config.yaml", encoding="utf-8") as f:
    config = yaml.safe_load(f)
import functions as fn
#project_root = Path.cwd().parent
#sys.path.append(str(project_root))

             
#Data raw folder path:
raw_folder = r"C:\Users\ziden\Desktop\Trainings\RNCP-Project\data\raw"
#Data clean folder path:
clean_folder = r"C:\Users\ziden\Desktop\Trainings\RNCP-Project\data\clean"

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [79]:
importlib.reload(fn)

<module 'functions' from 'C:\\Users\\ziden\\Desktop\\Trainings\\RNCP-Project\\functions.py'>

In [8]:
#-----------------------------------------------------------------------------
# 1. JOINING Population data with geo data: pop_geo_merge_df(population_df, geo_df)
#-----------------------------------------------------------------------------
#open Geographic file

geo_df_file_path = os.path.join(clean_folder, "insee_geo.csv")
geo_df = pd.read_csv(geo_df_file_path, sep=",", dtype={"insee_com": "str"}, encoding="latin1")

#open population file
population_df_file_path = os.path.join(clean_folder, "insee_com_population.csv")
population_df = pd.read_csv(population_df_file_path, sep=",", dtype={"insee_com": "str"}, encoding="latin1")

pop_geo_clean_df = fn.pop_geo_merge_df(population_df, geo_df)
pop_geo_clean_df.head()
pop_geo_clean_df.insee_com.nunique()

File 'geo_population_table.csv' is successfully saved to 'data/clean' folder.


34858

In [9]:
#-----------------------------------------------------------------------------
# 2. CLEANING ARCEP QOS FILE. function: clean_qos_df(raw_qos_df): "file1" in data/raw folder
#-----------------------------------------------------------------------------

clean_qos_df= fn.clean_qos_df("file1")
print(clean_qos_df.dtypes)


File '5G_qos_clean.csv' is successfully saved to 'data/clean' folder.
measure_id                             int64
acess_duration                       float64
bitrate_dl                           float64
bitrate_ul                           float64
date_start                    datetime64[us]
hour_start                            object
insee_com                             string
latitude_start                       float64
loaded_in_less_10_secondes           float64
loaded_in_less_5_secondes            float64
longitude_start                      float64
operator                                 str
quality_correct                      float64
quality_perfect                      float64
rsrp                                 float64
rsrq                                 float64
protocole                                str
situation                                str
techno_start                             str
terminal                                 str
url                           

In [10]:
#-----------------------------------------------------------------------------
# 3. CLEANING INSEE SITES FILE. file2 in "data/raw" folder ==>Corresponding fucntion name: clean_sites_file(file)
#-----------------------------------------------------------------------------

sites_clean_df = fn.clean_sites_file("file2")
sites_clean_df.columns

File 'insee_sites_clean.csv' is successfully saved to 'data/clean' folder.


Index(['site_id', 'code_op', 'nom_op', 'num_site', 'id_site_partage',
       'id_station_anfr', 'latitude', 'longitude', 'nom_reg', 'nom_dep',
       'insee_dep', 'nom_com', 'insee_com', 'site_4g', 'site_5g',
       'mes_4g_trim', 'date_ouverturecommerciale_5g', 'site_5g_700_m_hz',
       'site_5g_800_m_hz', 'site_5g_1800_m_hz', 'site_5g_2100_m_hz',
       'site_5g_3500_m_hz', 'operator_id'],
      dtype='str')

In [52]:
geo_po_clean = pd.read_csv(PROJECT_ROOT/config["data"]["clean"]["file1"], sep=(","), dtype={"insee_com":"str"}, encoding="latin1")
geo_po_clean.insee_com.nunique()


34858

In [69]:
#-----------------------------------------------------------------------------
# 4. ISOLATE distinct communes in which meausrements have been take, to project it on performance analysis
#-----------------------------------------------------------------------------
def common_unique_communes():
# unique communes in qos_raw file: total = 1614
qos_clean = pd.read_csv(PROJECT_ROOT/config["data"]["clean"]["file3"], sep=",", dtype={"insee_com":"str"},  encoding="latin1")
qos_clean_unique_communes = qos_clean.insee_com.unique()
print("Number of communes in raw qos files is: ", len(qos_clean_unique_communes))

#unique communes in sites_raw file total=20690
sites_clean = pd.read_csv(PROJECT_ROOT/config["data"]["clean"]["file2"], sep=(","), dtype={"insee_com":"str"}, encoding="utf-8")
sites_clean_unique_communes = sites_raw.insee_com.unique()
print("Number of communes in 'sites_clean' files is: ", len(sites_clean_unique_communes))

#unique communes in population file tota= 34858
pop_geo_clean = pd.read_csv(PROJECT_ROOT/config["data"]["clean"]["file1"], sep=(","), dtype={"insee_com":"str"}, encoding="latin1")
pop_geo_clean_unique_communes = population_raw["insee_com"].unique()
print("Number of communes in 'pop_geo_clean' files is: ", len(pop_geo_clean_unique_communes))

#find the common unique communes:
common_unique_communes = set(qos_clean_unique_communes) &  set(sites_clean_unique_communes) & set(pop_geo_clean_unique_communes)
print("Number of common communes is:", len(common_unique_communes))
#common_uniqe_communes
      


Number of communes in raw qos files is:  1486
Number of communes in 'sites_clean' files is:  1486
Number of communes in 'pop_geo_clean' files is:  34858
Number of common communes is: 1486


In [ ]:


unique_communes_in_insee_sites_df = sites_clean_df.insee_com.unique()
unique_communes_in_qos_df

#Determine the communes present in the 3 files: qos_df_clean, geo_pop_clean_df and sites_clean_df
print("Number of total communes in sites_clean_df is:", len(communes_in_insee_sites_df))
test_communes = set(communes_with_measurements) & set(communes_in_geo_po_file) & set(communes_in_insee_sites_df)
print("Number of communes in files: qos_df_clean, geo_pop_clean_df and sites_clean_df is:", len(test_communes))

#print("\nTest communes are:", test_communes)
#Save the list of test communes in data_clean folder
test_communes_series = pd.Series(list(test_communes))
test_communes_series.to_csv(os.path.join(clean_folder, "test_communes.csv"), index=False, encoding="utf-8")
print("file 'test_communes.csv' is successfully saved in 'data/clean' folder")

test_communes_series